In [1]:
import numpy as np
import random
import matplotlib.pyplot as plt

import os
import cv2
import albumentations as A

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import TensorDataset, DataLoader, random_split

In [2]:
seed = 42

torch.manual_seed(seed)
np.random.seed(seed)
random.seed(seed)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

## Data Loading

In [3]:
# Load Datasets
IMG_SIZE = (256, 256)
def load_images2(class_folder, img_size=IMG_SIZE, limit=100):
    count = 0
    X = []

    for file in os.listdir(class_folder):
        if count >= limit:
            break
        img_path = os.path.join(class_folder, file)

        # Read image (BGR)
        img = cv2.imread(img_path) 

        if img is None:
            continue  # skip corrupted files

        img = cv2.resize(img, img_size) # Resizing led to worse quality images with fewer keypoints

        X.append(img)
        count += 1

    return np.array(X)

In [4]:
train_folder = "Datasets/road/train"
test_folder = "Datasets/road/test"

train_img = []
train_label = []

image_files = sorted([f for f in os.listdir(train_folder) if f.endswith('_sat.jpg')])
mask_files = sorted([f for f in os.listdir(train_folder) if f.endswith('_mask.png')])

LIMIT = 1000 

for img_file, mask_file in zip(image_files[:LIMIT], mask_files[:LIMIT]):
    img = cv2.imread(os.path.join(train_folder, img_file))
    mask = cv2.imread(os.path.join(train_folder, mask_file), cv2.IMREAD_GRAYSCALE)
    
    if img is None or mask is None:
        continue
    
    img = cv2.resize(img, IMG_SIZE)
    mask = cv2.resize(mask, IMG_SIZE, interpolation=cv2.INTER_NEAREST)
    binary_mask = (mask > 127).astype(np.uint8)
    binary_mask = np.expand_dims(binary_mask, axis=0)

    train_img.append(img)
    train_label.append(binary_mask)

In [5]:
X_train = np.array(train_img)
Y_train = np.array(train_label)
X_test = load_images2("Datasets/road/test")

print("X train shape: ",X_train.shape)
print("Y Train shape: ",Y_train.shape)
print("X Test shape: ",X_test.shape)

print("Classes: ",np.unique(Y_train))

X train shape:  (1000, 256, 256, 3)
Y Train shape:  (1000, 1, 256, 256)
X Test shape:  (100, 256, 256, 3)
Classes:  [0 1]


In [6]:
num_pos = np.sum(Y_train == 1)
num_neg = np.sum(Y_train == 0)

pos_weight = num_neg / (num_pos + 1e-6) * 1.5

In [7]:
# Normalizing pixel values to prevent exploding gradients
X_train = (X_train / 255.0).astype(np.float32)
X_test = (X_test / 255.0).astype(np.float32)

## Dataset Preparation

- Data converted to PyTorch tensors
- Train-validation split applied (80-20)
- DataLoader used for batching and shuffling

In [8]:
# Convert to Pytorch Format

X_train_torch = torch.tensor(X_train, dtype=torch.float32).permute(0, 3, 1, 2)
Y_train_torch = torch.tensor(Y_train, dtype=torch.float32)

X_test_torch = torch.tensor(X_test, dtype=torch.float32).permute(0, 3, 1, 2)

print("Train X shape:", X_train_torch.shape)
print("Train Y shape:", Y_train_torch.shape)
print("Test X shape:", X_test_torch.shape)

Train X shape: torch.Size([1000, 3, 256, 256])
Train Y shape: torch.Size([1000, 1, 256, 256])
Test X shape: torch.Size([100, 3, 256, 256])


In [9]:
# Create DataLoader

train_dataset = TensorDataset(X_train_torch, Y_train_torch)

train_size = int(0.8 * len(train_dataset))
val_size = len(train_dataset) - train_size

train_dataset, val_dataset = random_split(
    train_dataset,
    [train_size, val_size],
    generator=torch.Generator().manual_seed(42)
)

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=4)

## Model Architecture

In [10]:
def double_conv(in_c, out_c):
    return nn.Sequential(
        nn.Conv2d(in_c, out_c, kernel_size=3, padding=1),
        nn.BatchNorm2d(out_c),
        nn.ReLU(inplace=True),
        nn.Conv2d(out_c, out_c, kernel_size=3, padding=1),
        nn.BatchNorm2d(out_c),
        nn.ReLU(inplace=True)
    )

In [11]:
class UNet(nn.Module):
    def __init__(self, in_channels=3, out_channels=1):
        super(UNet, self).__init__()

        # Encoder (Downsampling)
        self.enc1 = double_conv(in_channels, 64)
        self.enc2 = double_conv(64, 128)
        #self.enc3 = double_conv(64,128)
        self.pool = nn.MaxPool2d(2)
        
        # Bottleneck
        self.bottleneck = nn.Sequential(
            double_conv(128, 256),
            nn.Dropout2d(0.3)
        )
    
        # Decoder (Upsampling)
        #self.up3 = nn.ConvTranspose2d(256, 128, 2, 2)
        #self.dec3 = double_conv(256, 128)
        
        self.up2 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        self.dec2 = double_conv(256, 128) 
        
        self.up1 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.dec1 = double_conv(128, 64) 
        
        # Output layer
        self.final_conv = nn.Conv2d(64, out_channels, kernel_size=1)

    def forward(self, x):
        # Encoder
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        #e3 = self.enc3(self.pool(e2))
        
        # Bottleneck
        b = self.bottleneck(self.pool(e2))
        
        # Decoder with Skip Connections
        #d3 = self.up3(b)
        #d3 = torch.cat([d3, e3], dim=1)
        #d3 = self.dec3(d3)

        d2 = self.up2(b)
        d2 = torch.cat([d2, e2], dim=1) # Concatenate along channel dim
        d2 = self.dec2(d2)
        
        d1 = self.up1(d2)
        d1 = torch.cat([d1, e1], dim=1)
        d1 = self.dec1(d1)
        
        return self.final_conv(d1)

# Training

In [12]:
# Training setup

model = UNet()
 
pos_weight_tensor = torch.tensor(pos_weight, dtype=torch.float32)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
pos_weight_tensor = pos_weight_tensor.to(device)

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight_tensor)
optimizer = torch.optim.Adam(model.parameters(), lr=0.0003)

In [13]:
def dice_score(pred, target):
    pred = torch.sigmoid(pred)
    intersection = (pred * target).sum(dim=(1,2,3))
    union = pred.sum(dim=(1,2,3)) + target.sum(dim=(1,2,3))

    dice = (2 * intersection + 1e-6) / (union + 1e-6)
    return dice.mean()

In [14]:
def dice_loss(pred, target):
    pred = torch.sigmoid(pred)

    intersection = (pred * target).sum(dim=(1,2,3))
    union = pred.sum(dim=(1,2,3)) + target.sum(dim=(1,2,3))

    dice = (2 * intersection + 1e-6) / (union + 1e-6)
    return 1 - dice.mean()

In [15]:
def morph_batch(pred_batch):
    pred_batch = pred_batch.cpu().numpy().astype(np.uint8)

    processed = []

    kernel = np.ones((3,3), np.uint8)

    for i in range(pred_batch.shape[0]):  # loop over batch
        mask = pred_batch[i][0]

        mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)
        mask = cv2.dilate(mask, kernel, iterations=1)

        processed.append(mask)

    return np.array(processed)[:, None, :, :]  # restore shape (B,1,H,W)

In [ ]:
# Training Loop

epochs = 20
val_dices = []

for epoch in range(epochs):

    # TRAINING
    model.train()   # sets model to training mode -> affects dropout(enabled), BatchNorm
    correct = 0
    total = 0
    running_loss = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()   # clears old gradients
        outputs = model(images)

        bce = criterion(outputs, labels)
        d_loss = dice_loss(outputs, labels)

        loss = 0.5 * bce + 0.5 * d_loss

        loss.backward()     # Gradients computed
        optimizer.step()    # Gradients updated

        running_loss += loss.item()

    # VALIDATION 
    model.eval()        # Sets model to evaluation mode -> dropout(disabled), BatchNorm uses previously stored mean and variance instead of computing a new one
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        raw_dice = 0
        #morph_dice = 0
        mean_preds = []

        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            probs = torch.sigmoid(outputs)
            raw_dice += dice_score(outputs, labels)
            # Apply morphology
            #preds_np = morph_batch(preds)
            # Convert back to tensor
            #morphed_preds = torch.tensor(preds_np, dtype=torch.float32).to(device)
            # morph_dice += dice_score(morphed_preds, labels)
            mean_preds.append(probs.mean().item())

        val_dice_raw = raw_dice / len(val_loader)
        #val_dice_morph = morph_dice / len(val_loader)

    val_dices.append(val_dice_raw)

    print(f"Epoch {epoch+1}, Val Dice: {val_dice_raw:.4f}, Mean prediction: {np.mean(mean_preds):.4f}")

Epoch 1, Val Dice: 0.1297, Mean prediction: 0.4307
Epoch 2, Val Dice: 0.1627, Mean prediction: 0.3030
Epoch 3, Val Dice: 0.1983, Mean prediction: 0.2649
Epoch 4, Val Dice: 0.2188, Mean prediction: 0.2343
Epoch 5, Val Dice: 0.2253, Mean prediction: 0.2310
Epoch 6, Val Dice: 0.2651, Mean prediction: 0.1737
Epoch 7, Val Dice: 0.1951, Mean prediction: 0.3001
Epoch 8, Val Dice: 0.2784, Mean prediction: 0.1971
Epoch 9, Val Dice: 0.2562, Mean prediction: 0.2275
Epoch 10, Val Dice: 0.3082, Mean prediction: 0.1674
Epoch 11, Val Dice: 0.3025, Mean prediction: 0.1604
Epoch 12, Val Dice: 0.2900, Mean prediction: 0.1909
Epoch 13, Val Dice: 0.2975, Mean prediction: 0.1885


In [ ]:
img = images[0].detach().cpu().permute(1,2,0).numpy()
img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
prob = probs[0][0].detach().cpu().numpy()
pred = (prob > 0.5).astype(np.uint8)
gt = labels[0][0].detach().cpu().numpy()

fig, axs = plt.subplots(1,4, figsize=(16,4))

axs[0].imshow(img)
axs[0].set_title("Image")

axs[1].imshow(prob, cmap='gray')
axs[1].set_title("Prob Map")

axs[2].imshow(pred, cmap='gray')
axs[2].set_title("Binary Pred")

axs[3].imshow(gt, cmap='gray')
axs[3].set_title("Ground Truth")

for ax in axs:
    ax.axis('off')

plt.show()

In [ ]:
def segment_image(model, image, device):
    model.eval()

    img = cv2.resize(image, (256, 256))
    img = img / 255.0
    img = img.astype(np.float32)

    img_t = torch.tensor(img).permute(2,0,1).unsqueeze(0).to(device)

    with torch.no_grad():
        output = model(img_t)
        prob = torch.sigmoid(output)[0,0].cpu().numpy()

    return prob

In [ ]:
topdown_image = cv2.imread("topdown_image.png")
prob = segment_image(model, topdown_image, device)

mask = (prob > 0.5).astype(np.uint8) * 255  # VERY IMPORTANT

mask = cv2.resize(mask, 
                  (topdown_image.shape[1], topdown_image.shape[0]),
                  interpolation=cv2.INTER_NEAREST)

cv2.imwrite("segmented_mask.png", mask)

## Results: 
With 500 images and 3 Convolutional layers:     

Epoch 1, Val Dice: 0.1538, Mean prediction: 0.4241  
Epoch 2, Val Dice: 0.1291, Mean prediction: 0.4990  
Epoch 3, Val Dice: 0.1187, Mean prediction: 0.5575  
Epoch 4, Val Dice: 0.1862, Mean prediction: 0.3642  
Epoch 5, Val Dice: 0.1494, Mean prediction: 0.4350  
Epoch 6, Val Dice: 0.1916, Mean prediction: 0.3450  
Epoch 7, Val Dice: 0.2750, Mean prediction: 0.2285  
Epoch 8, Val Dice: 0.2151, Mean prediction: 0.3174  
Epoch 9, Val Dice: 0.2633, Mean prediction: 0.2683  
Epoch 10, Val Dice: 0.1594, Mean prediction: 0.4091     
Epoch 11, Val Dice: 0.3641, Mean prediction: 0.1739          
Epoch 12, Val Dice: 0.2436, Mean prediction: 0.2939       
Epoch 13, Val Dice: 0.2035, Mean prediction: 0.3366     
`Epoch 14, Val Dice: 0.3829, Mean prediction: 0.1740 `    
Epoch 15, Val Dice: 0.3050, Mean prediction: 0.2325      
Epoch 16, Val Dice: 0.3425, Mean prediction: 0.1914        
Epoch 17, Val Dice: 0.3424, Mean prediction: 0.1872        
Epoch 18, Val Dice: 0.3216, Mean prediction: 0.2145     
Epoch 19, Val Dice: 0.2726, Mean prediction: 0.2453     
Epoch 20, Val Dice: 0.2649, Mean Prediction: 0.2884          

With 1500 images and 2 Convolutional layers: 


# To Try:
* Add Visualizations
* Remove noise using size, aspect ratio
* Use morphology for cleaning 